# 08. Grounded Answer
## Context 안에서만 답하고 출처 표시하기

### 이번 실습의 위치

```text
07 Context Engineering
        ↓
Compact Context
        ↓
08 Grounded Answer
        ↓
근거 기반 답변 + 출처 표시
        ↓
09 Evaluation
```

이번 실습의 핵심은 세 가지이다.

1. 제공된 Context 안에서만 답한다.
2. 답변에 어떤 근거를 사용했는지 표시한다.
3. 근거가 없으면 억지로 답하지 않는다.

> **Grounded Answer는 LLM의 지식이 아니라 제공된 근거에 기반해 답변하는 방식이다.**


# 1. Grounded Answer란?

일반적인 LLM은 자신이 알고 있는 지식을 섞어 답할 수 있다.

RAG에서는 다음처럼 답변 범위를 제한하는 것이 중요하다.

```text
Question
   +
Context
   ↓
LLM
   ↓
Context 안에서만 답변
```

즉, 검색된 근거를 벗어나지 않도록 프롬프트로 제한한다.


# 2. 환경 설정

07번에서 만든 Compact Context를 바로 연결하는 것이 실제 흐름이다.

하지만 이 노트북을 독립적으로 실행할 수 있도록,
간단한 예제 Context를 직접 준비한다.


In [1]:
# 상위 폴더의 common_config.py를 불러오기 위한 경로 설정
import sys
sys.path.append("..")

# LLM 연결 함수
from common_config import llm_connect


# 답변 생성용 LLM
# temperature=0으로 설정해 답변이 불필요하게 흔들리지 않도록 한다.
llm = llm_connect(
    temperature=0,
    max_tokens=400,
)


# 3. 출처 ID가 포함된 Context 준비

각 근거 앞에 `[S1]`, `[S2]`처럼 간단한 Source ID를 붙인다.

```text
[S1] 기업 내부에서 AI 에이전트 활용이 증가할 것으로 전망된다.

[S2] 사람과 AI 에이전트가 협업하는 업무 구조가 확대될 수 있다.
```

LLM은 이 ID를 답변에 그대로 사용하면 된다.


In [2]:
# 예제 질문
question = "기업 내부에서 AI 에이전트는 어떤 방식으로 활용될 것으로 전망되나요?"


# 실제 서비스에서는 07번에서 만든 Compact Context를 사용한다.
# 여기서는 실습을 쉽게 하기 위해 간단한 근거를 직접 준비한다.
context = """
[S1]
기업 내부에서 AI 에이전트를 활용한 문서 처리,
고객 지원, 운영 자동화가 증가할 것으로 전망된다.

[S2]
사람과 AI 에이전트가 협업하는 형태의 업무 구조가
일부 영역에서 확대될 가능성이 있다.
"""

print("[Context]")
print(context)


[Context]

[S1]
기업 내부에서 AI 에이전트를 활용한 문서 처리,
고객 지원, 운영 자동화가 증가할 것으로 전망된다.

[S2]
사람과 AI 에이전트가 협업하는 형태의 업무 구조가
일부 영역에서 확대될 가능성이 있다.



# 4. Grounded Answer 생성

프롬프트 규칙은 복잡할 필요가 없다.

다음 세 가지만 명확하면 된다.

```text
1. Context에 있는 내용만 사용한다.
2. 사용한 근거 뒤에 [S1], [S2]처럼 표시한다.
3. 근거가 없으면 답할 수 없다고 말한다.
```


In [3]:
# Grounded Answer용 프롬프트
prompt = f"""
아래 Context만 사용해서 질문에 답하세요.

규칙:
- Context에 있는 내용만 사용하세요.
- 사용한 근거 뒤에 [S1], [S2]처럼 출처를 표시하세요.
- Context에 없는 내용은 추측하지 마세요.
- 근거가 없으면
  "제공된 근거만으로는 답변하기 어렵습니다."
  라고 답하세요.

질문:
{question}

Context:
{context}
"""

# LLM에게 질문과 Context를 함께 전달한다.
response = llm.invoke(prompt)

# 생성된 Grounded Answer 확인
print("[질문]")
print(question)

print("\n[Grounded Answer]")
print(response.content)


[질문]
기업 내부에서 AI 에이전트는 어떤 방식으로 활용될 것으로 전망되나요?

[Grounded Answer]
기업 내부에서 AI 에이전트는 문서 처리, 고객 지원, 운영 자동화에 활용되는 방식이 증가할 것으로 전망됩니다. 또한 일부 영역에서는 사람과 AI 에이전트가 협업하는 업무 구조가 확대될 가능성이 있습니다. [S1][S2]


# 5. 근거 없는 질문 테스트

Grounded RAG에서는 **모르면 답하지 않는 것**도 중요하다.

현재 Context에는 GPT-5의 벤치마크 점수에 대한 정보가 없다.

따라서 정상적인 답변은 다음과 같아야 한다.

```text
제공된 근거만으로는 답변하기 어렵습니다.
```


In [4]:
unknown_question = "GPT-5의 대표적인 벤치마크 점수는 얼마인가요?"

prompt = f"""
아래 Context만 사용해서 질문에 답하세요.

규칙:
- Context에 있는 내용만 사용하세요.
- Context에 없는 내용은 추측하지 마세요.
- 근거가 없으면
  "제공된 근거만으로는 답변하기 어렵습니다."
  라고 답하세요.

질문:
{unknown_question}

Context:
{context}
"""

response = llm.invoke(prompt)

print("[근거 없는 질문]")
print(unknown_question)

print("\n[답변]")
print(response.content)


[근거 없는 질문]
GPT-5의 대표적인 벤치마크 점수는 얼마인가요?

[답변]
제공된 근거만으로는 답변하기 어렵습니다.


# 6. 왜 Citation이 필요한가?

Grounded Answer에서 `[S1]`, `[S2]` 같은 출처 표시는 다음 의미가 있다.

```text
답변
  ↓
어떤 근거를 사용했는가?
  ↓
[S1], [S2]
```

사용자는 답변의 근거를 확인할 수 있고,
개발자는 어떤 문서가 실제 답변에 사용되었는지 추적할 수 있다.

이번 실습에서는 Citation Validation 코드를 별도로 구현하지 않는다.

실무에서는 존재하지 않는 Source ID를 LLM이 생성했는지 검증하는 단계를 추가할 수 있다.


# 7. 07번과의 연결

07번에서 만든 결과를 08번에 연결하면 흐름은 다음과 같다.

```text
07 Context Engineering
검색 문서
   ↓
필요한 근거만 추출
   ↓
Compact Context
   ↓
[S1], [S2]와 같은 Source ID 부여
   ↓
08 Grounded Answer
   ↓
근거 기반 답변
```

즉,

> **07번은 어떤 내용을 Context에 넣을지 결정하고,  
> 08번은 그 Context 안에서만 답하도록 만드는 단계이다.**


# 8. 핵심 정리

## 오늘 기억할 것

```text
Context
   ↓
Grounded Prompt
   ↓
LLM
   ↓
Answer + Citation
```

### 핵심 원칙

1. **Context Only**  
   제공된 근거 안에서만 답한다.

2. **Citation**  
   사용한 근거를 `[S1]`, `[S2]`처럼 표시한다.

3. **No Answer**  
   근거가 없으면 추측하지 않는다.

---

## 다음 실습

```text
09 Evaluation
```

09번에서는 검색 결과와 답변이 실제로 얼마나 좋은지 평가하는 방법으로 넘어간다.
